# Hướng Dẫn Thực Nghiệm Colab(backbone)

> **Lưu ý:**
> 1. **Runtime GPU:** Chọn **T4 GPU** hoặc **A100** cho nó nhanh :))
> 2. **Lối tắt Drive:** Hãy tạo lối tắt (Shortcut) thư mục `KLTN-2026-testingNtraining` vào `My Drive` của bạn trước khi chạy.

## Bước 1: Kết nối Google Drive & Kiểm tra 

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

SHARED_DRIVE = "/content/drive/MyDrive/KLTN-2026-testingNtraining"

if not os.path.exists(SHARED_DRIVE):
    print("❌ CHƯA TÌM THẤY THƯ MỤC CHUNG!")
    print("👉 Hãy vào Google Drive web -> 'Được chia sẻ với tôi' -> Chuột phải vào 'KLTN-2026-testingNtraining' -> 'Thêm lối tắt vào Drive' -> chọn 'My Drive' rồi chạy lại cell này.")
else:
    print(f"✅ Đã kết nối thành công với thư mục chung: {SHARED_DRIVE}")
    os.makedirs(f"{SHARED_DRIVE}/runs/baseline", exist_ok=True)
    os.makedirs(f"{SHARED_DRIVE}/runs/cbam_backbone", exist_ok=True)
    os.makedirs(f"{SHARED_DRIVE}/runs/evaluation", exist_ok=True)
    print("✅ Các thư mục runs/baseline, runs/cbam_backbone, runs/evaluation đã sẵn sàng!")

## Bước 2: Clone Source Code từ GitHub & Cài đặt Thư viện

In [ ]:
!git clone https://github.com/mizzhau/yolo11n-cbam-mvtec-defect-detection.git
%cd yolo11n-cbam-mvtec-defect-detection
!pip install -q ultralytics albumentations tabulate

## Bước 3: Ghép 5 Part Dữ Liệu & Giải Nén Lên SSD Colab

In [ ]:
!cat /content/drive/MyDrive/KLTN-2026-testingNtraining/dataset/mvtec_augmented_2.zip.00* > /content/archive_temp.zip
!mkdir -p data/processed/split_70_15_15_augmented
!7z x /content/archive_temp.zip -odata/processed/split_70_15_15_augmented -y > /dev/null
!rm -f /content/archive_temp.zip

!unzip -q data/processed/split_70_15_15_augmented/mvtec_augmented.zip -d data/processed/split_70_15_15_augmented/
!rm -f data/processed/split_70_15_15_augmented/mvtec_augmented.zip

!ls -la data/processed/split_70_15_15_augmented

## Bước 4: Thực Nghiệm

### Cell 4A: Huấn Luyện BASELINE

In [ ]:
!python src/training/train_baseline.py \
    --data configs/data/mvtec_70_15_15_augmented.yaml \
    --epochs 20 \
    --batch 16 \
    --imgsz 640 \
    --save_period 1 \
    --project /content/drive/MyDrive/KLTN-2026-testingNtraining/runs/baseline \
    --name train

### Cell 4B: Huấn Luyện MÔ HÌNH CHÍNH

In [ ]:
!python src/training/train_cbam.py \
    --data configs/data/mvtec_70_15_15_augmented.yaml \
    --epochs 20 \
    --batch 16 \
    --imgsz 640 \
    --save_period 1 \
    --project /content/drive/MyDrive/KLTN-2026-testingNtraining/runs/cbam_backbone \
    --name train

## Bước 5: Đánh Giá So Sánh & Xuất Biểu Đồ Tự Động Về Drive Chung


In [ ]:
!python src/evaluation/plot_comparison.py \
    --baseline_dir /content/drive/MyDrive/KLTN-2026-testingNtraining/runs/baseline/train \
    --cbam_dir /content/drive/MyDrive/KLTN-2026-testingNtraining/runs/cbam_backbone/train \
    --output_dir /content/drive/MyDrive/KLTN-2026-testingNtraining/runs/evaluation

print("\nHoàn tất đánh giá so sánh! Biểu đồ và báo cáo đã được lưu tại:")
!ls -la /content/drive/MyDrive/KLTN-2026-testingNtraining/runs/evaluation